In [ ]:
from dataclasses import dataclass
from torch import Tensor

@dataclass
class movie:
    ryear:float
    title:str
    runtime:int
    genres:set
    overview:str
    rating:float
    meta_score:float
    directors:set
    stars:set
    gross:float



In [ ]:
#load SBERT


from sentence_transformers import SentenceTransformer,util
sbert = SentenceTransformer('all-MiniLM-L6-v2')# load the pretrained model.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
import csv

#load movies from the input file
def load_movies(input_file:str):


    title_index={}# title to index
    movies=[]
    with open(input_file) as f:
        overviews=[]

        for row in csv.DictReader(f): # for each movie

            #make a new movie object
            new_mv=movie(int(row['Released_Year']) if row['Released_Year'].isnumeric() else None,
                         row['Series_Title'],
                         int(row['Runtime'][:row['Runtime'].find(' ')]),
                         set([x.strip() for x in row['Genre'].split(',')]),
                         row['Overview'],
                         float(row['IMDB_Rating']),
                         float(row['Meta_score'][:row['Meta_score'].find(' ')]) if row['Meta_score']!='' else None,
                         set([x.strip() for x in row['Director'].split(',')]),
                         set([row[x] for x in ['Star1','Star2','Star3','Star4']]),
                         float(row['Gross'].replace(',','')) if row['Gross']!='' else None)

            #store the overview separately
            overviews.append(row['Overview'])

            #update the index
            title_index[row['Series_Title']]=len(movies)
            movies.append(new_mv)

    #encode all overviews
    embedded=sbert.encode(overviews,convert_to_tensor=True)

    #Compute cosine-similarities
    sim_matrix = util.cos_sim(embedded, embedded)

    return movies,sim_matrix,title_index


In [ ]:
movies,sim_matrix,title_index=load_movies('imdb_top_1000.csv')

In [ ]:
sim_matrix[10,198]

tensor(-0.0371)

In [ ]:
import numpy as np

#movie-to-movie similarity
def sim_v2(title1:str,
           title2:str,
           movies:list,
           weights:dict,
           sim_matrix:Tensor,
           title_index:dict)->float:

    mid1,mid2=title_index[title1],title_index[title2]# get the movie ids (idnexes)
    m1=movies[mid1] # get the movie objects
    m2=movies[mid2]

    scores=dict() # stores a the score for each factor from the weights dict

    #star jacard
    scores['star']=len(m1.stars.intersection(m2.stars))/len(m1.stars.union(m2.stars))

    #director jaccard
    scores['director']=len(m1.directors.intersection(m2.directors))/len(m1.directors.union(m2.directors))

    #genre jaccard
    scores['genre']=len(m1.genres.intersection(m2.genres))/len(m1.genres.union(m2.genres))

    # release year diff
    try:
        scores['ryear']=abs(m1.ryear-m2.ryear)/100
    except:
        scores['ryear']=0

    #cosine sim for overviews
    scores['overview']=sim_matrix[mid1,mid2].numpy()

    #normalized candidate rating
    scores['rating']=m1.rating/10

    #create the sim dict
    factors={x:round(scores[x]*weights[x],2) for x in scores}

    #sort factors by sim
    sorted_factors=[factor for factor in sorted(factors.items(), key=lambda x:x[1],reverse=True) if factor[1]>0]

    #return overall score and explanations
    return round(np.sum(list(factors.values())),2),sorted_factors

In [ ]:

sim_v2('The Dark Knight',
       'Batman Begins',
       movies,
       {'ryear':1, 'genre':1, 'director':1, 'rating':1, 'star':1, 'overview': 1},
       sim_matrix,
       title_index)

(np.float64(3.08),
 [('director', 1.0),
  ('rating', 0.9),
  ('overview', np.float32(0.57)),
  ('star', 0.33),
  ('genre', 0.25),
  ('ryear', 0.03)])

In [ ]:
def recommend(input_title:str,
              k:int,
              movies:dict,
              weights:dict,
              sim_matrix:Tensor,
              title_index:dict
              )->list:

    results={} # recommendations

    for candidate in movies: # for each candidate

        #get the similarity and the explanation
        my_sim,my_exp=sim_v2(candidate.title,input_title,movies, weights,sim_matrix,title_index)

        #remember
        results[candidate.title]=(my_sim,my_exp)

    #sort, slice, return
    return sorted(results.items(),key=lambda x:x[1][0],reverse=True)[:k]

In [ ]:

weights={'ryear':1, 'genre':1, 'director':1, 'rating':1, 'star':1, 'overview': 1}

recommend('Toy Story',5,movies,weights,sim_matrix,title_index)

[('Toy Story',
  (np.float64(4.83),
   [('star', 1.0),
    ('director', 1.0),
    ('genre', 1.0),
    ('overview', np.float32(1.0)),
    ('rating', 0.83)])),
 ('Toy Story 2',
  (np.float64(3.53),
   [('director', 1.0),
    ('genre', 1.0),
    ('rating', 0.79),
    ('overview', np.float32(0.37)),
    ('star', 0.33),
    ('ryear', 0.04)])),
 ('Toy Story 4',
  (np.float64(2.75),
   [('genre', 1.0),
    ('rating', 0.78),
    ('overview', np.float32(0.4)),
    ('star', 0.33),
    ('ryear', 0.24)])),
 ('Toy Story 3',
  (np.float64(2.65),
   [('genre', 1.0),
    ('rating', 0.82),
    ('overview', np.float32(0.35)),
    ('star', 0.33),
    ('ryear', 0.15)])),
 ('Klaus',
  (np.float64(2.3),
   [('genre', 1.0),
    ('rating', 0.82),
    ('ryear', 0.24),
    ('overview', np.float32(0.24))]))]

In [ ]:
# Part 1: Bot Simulation
# ======================
# 10 bots, each with:
#   - 1 core (seed) movie — the bot's favorite movie
#   - attribute weights — how much the bot cares about each similarity factor (0 to 1)
#   - a noise percentage (5-20%) — probability of randomly flipping a decision
#   - a threshold_percentile — how picky the bot is
#     e.g. percentile 50 = bot likes top 50% of movies (generous)
#          percentile 90 = bot only likes top 10% (strict)
#
# Each bot rates 500 different movies → total 5000 ratings
# Label: 1 = YES (positive), 0 = NO (negative)

import random
import pandas as pd

@dataclass
class Bot:
    bot_id: int
    core_movie: str        # the bot's favorite movie
    weights: dict          # how much it cares about each factor
    noise_pct: float       # chance of flipping a decision randomly
    threshold_percentile: float  # how picky (higher = stricter)

# Define 10 bots — each with a different favorite movie and personality
bot_configs = [
    # Bot 0 — Loves classic crime drama, strict (top 15% liked)
    Bot(0, 'The Godfather',
        {'ryear': 0.9, 'genre': 0.6, 'director': 1.0, 'rating': 0.8, 'star': 0.5, 'overview': 0.3},
        0.08, 85),

    # Bot 1 — Loves prison drama, very generous (top 55% liked)
    Bot(1, 'The Shawshank Redemption',
        {'ryear': 0.5, 'genre': 0.8, 'director': 0.4, 'rating': 1.0, 'star': 0.3, 'overview': 0.6},
        0.20, 45),

    # Bot 2 — Loves feel-good drama, moderate (top 25% liked)
    Bot(2, 'Forrest Gump',
        {'ryear': 0.4, 'genre': 0.8, 'director': 0.6, 'rating': 0.7, 'star': 0.8, 'overview': 0.9},
        0.10, 75),

    # Bot 3 — Loves sci-fi/space, generous (top 40% liked)
    Bot(3, 'Interstellar',
        {'ryear': 0.3, 'genre': 1.0, 'director': 0.8, 'rating': 0.5, 'star': 0.7, 'overview': 0.4},
        0.10, 60),

    # Bot 4 — Loves horror/thriller, moderate-strict (top 20% liked)
    Bot(4, 'Psycho',
        {'ryear': 0.7, 'genre': 1.0, 'director': 0.9, 'rating': 0.4, 'star': 0.2, 'overview': 0.5},
        0.12, 80),

    # Bot 5 — Loves war movies, strict (top 12% liked)
    Bot(5, 'Saving Private Ryan',
        {'ryear': 0.6, 'genre': 0.7, 'director': 1.0, 'rating': 0.9, 'star': 0.5, 'overview': 0.8},
        0.07, 88),

    # Bot 6 — Loves romance/epic, very strict (top 8% liked)
    Bot(6, 'Titanic',
        {'ryear': 0.4, 'genre': 0.9, 'director': 1.0, 'rating': 0.7, 'star': 0.6, 'overview': 0.8},
        0.05, 92),

    # Bot 7 — Loves comedy, very generous (top 60% liked)
    Bot(7, 'Some Like It Hot',
        {'ryear': 0.3, 'genre': 1.0, 'director': 0.5, 'rating': 0.4, 'star': 0.6, 'overview': 0.3},
        0.18, 40),

    # Bot 8 — Loves animation, moderate (top 30% liked)
    Bot(8, 'Coco',
        {'ryear': 0.2, 'genre': 1.0, 'director': 0.7, 'rating': 0.6, 'star': 0.9, 'overview': 0.5},
        0.15, 70),

    # Bot 9 — Loves superhero/action, extremely strict (top 5% liked)
    Bot(9, 'Spider-Man: Into the Spider-Verse',
        {'ryear': 0.8, 'genre': 0.5, 'director': 1.0, 'rating': 0.6, 'star': 0.3, 'overview': 0.7},
        0.06, 95),
]

# Print summary of all 10 bots
for b in bot_configs:
    approx_like = 100 - b.threshold_percentile
    print(f"Bot {b.bot_id}: core='{b.core_movie}', noise={b.noise_pct:.0%}, "
          f"percentile={b.threshold_percentile} (likes ~top {approx_like}%)")

Bot 0: core='The Godfather', noise=8%, percentile=85 (likes ~top 15%)
Bot 1: core='The Shawshank Redemption', noise=20%, percentile=45 (likes ~top 55%)
Bot 2: core='Forrest Gump', noise=10%, percentile=75 (likes ~top 25%)
Bot 3: core='Interstellar', noise=10%, percentile=60 (likes ~top 40%)
Bot 4: core='Psycho', noise=12%, percentile=80 (likes ~top 20%)
Bot 5: core='Saving Private Ryan', noise=7%, percentile=88 (likes ~top 12%)
Bot 6: core='Titanic', noise=5%, percentile=92 (likes ~top 8%)
Bot 7: core='Some Like It Hot', noise=18%, percentile=40 (likes ~top 60%)
Bot 8: core='Coco', noise=15%, percentile=70 (likes ~top 30%)
Bot 9: core='Spider-Man: Into the Spider-Verse', noise=6%, percentile=95 (likes ~top 5%)


In [ ]:
def generate_bot_ratings(bot_configs, movies, sim_matrix, title_index, ratings_per_bot=500, seed=42):

    random.seed(seed) # fix randomness so results are reproducible
    np.random.seed(seed)

    all_rows = [] # will store every rating as a dictionary

    for bot in bot_configs: # for each of the 10 bots

        # STEP 1: compute similarity of ALL 999 movies to this bot's core movie
        all_scores = []
        for m in movies:
            if m.title == bot.core_movie: # skip the core movie itself
                continue
            score, _ = sim_v2(m.title, bot.core_movie, movies, bot.weights, sim_matrix, title_index)
            all_scores.append(score) # collect every similarity score

        # STEP 2: calculate the actual threshold from the percentile
        # e.g. percentile=60 means: find the score where 60% of movies are BELOW it
        # so only the top 40% of movies will pass the threshold
        threshold = np.percentile(all_scores, bot.threshold_percentile)

        print(f"Bot {bot.bot_id} ({bot.core_movie}): "
              f"scores range [{min(all_scores):.2f}, {max(all_scores):.2f}], "
              f"percentile {bot.threshold_percentile} → threshold = {threshold:.2f}")

        # STEP 3: sample 500 random movies and rate them
        candidates = [m for m in movies if m.title != bot.core_movie] # all movies except core
        sampled = random.sample(candidates, min(ratings_per_bot, len(candidates))) # pick 500 randomly

        for candidate in sampled: # for each of the 500 sampled movies

            # compute similarity between this candidate and the bot's core movie
            score, _ = sim_v2(candidate.title, bot.core_movie,
                              movies, bot.weights, sim_matrix, title_index)

            # clean decision: if similarity >= threshold → YES (1), otherwise → NO (0)
            clean_label = 1 if score >= threshold else 0

            # apply noise: with probability = noise_pct, FLIP the decision
            # this simulates human unpredictability
            if random.random() < bot.noise_pct:
                noisy_label = 1 - clean_label # flip: YES becomes NO, NO becomes YES
            else:
                noisy_label = clean_label # keep the original decision

            # save this rating
            all_rows.append({
                'bot_id': bot.bot_id,
                'core_movie': bot.core_movie,
                'candidate_movie': candidate.title,
                'similarity': score,
                'threshold': round(threshold, 2),
                'label_clean': clean_label, # the decision WITHOUT noise
                'label': noisy_label # the FINAL decision (after noise)
            })

    df = pd.DataFrame(all_rows) # convert to a pandas DataFrame
    return df

In [ ]:
# Run the function: 10 bots x 500 movies each = 5000 total ratings
ratings_df = generate_bot_ratings(bot_configs, movies, sim_matrix, title_index, ratings_per_bot=500)

print(f"Total ratings: {len(ratings_df)}") # should be 5000
print(f"Columns: {list(ratings_df.columns)}")
print()
ratings_df.head(10) # show the first 10 rows

Bot 0 (The Godfather): scores range [0.66, 2.71], percentile 85 → threshold = 1.32
Bot 1 (The Shawshank Redemption): scores range [0.77, 2.00], percentile 45 → threshold = 1.21
Bot 2 (Forrest Gump): scores range [0.51, 1.98], percentile 75 → threshold = 1.08
Bot 3 (Interstellar): scores range [0.38, 1.79], percentile 60 → threshold = 0.74
Bot 4 (Psycho): scores range [0.29, 2.07], percentile 80 → threshold = 0.81
Bot 5 (Saving Private Ryan): scores range [0.63, 2.30], percentile 88 → threshold = 1.30
Bot 6 (Titanic): scores range [0.46, 1.89], percentile 92 → threshold = 1.39
Bot 7 (Some Like It Hot): scores range [0.31, 1.44], percentile 40 → threshold = 0.49
Bot 8 (Coco): scores range [0.41, 1.73], percentile 70 → threshold = 0.70
Bot 9 (Spider-Man: Into the Spider-Verse): scores range [0.46, 1.57], percentile 95 → threshold = 1.21
Total ratings: 5000
Columns: ['bot_id', 'core_movie', 'candidate_movie', 'similarity', 'threshold', 'label_clean', 'label']



,bot_id,core_movie,candidate_movie,similarity,threshold,label_clean,label
0,0,The Godfather,The Game,1.06,1.32,0,0
1,0,The Godfather,Per qualche dollaro in più,0.79,1.32,0,0
2,0,The Godfather,La vita è bella,1.14,1.32,0,0
3,0,The Godfather,Flipped,1.11,1.32,0,0
4,0,The Godfather,Zerkalo,0.96,1.32,0,0
5,0,The Godfather,"Crna macka, beli macor",1.13,1.32,0,0
6,0,The Godfather,Mary and Max,1.16,1.32,0,0
7,0,The Godfather,El secreto de sus ojos,1.21,1.32,0,0
8,0,The Godfather,Harry Potter and the Deathly Hallows: Part 1,1.00,1.32,0,0
9,0,The Godfather,Idi i smotri,1.03,1.32,0,0


In [ ]:
# Summary table: for each bot show how many YES vs NO ratings it gave
print("=" * 100)
print(f"{'Bot':>4} | {'Core Movie':<28} | {'Pctl':>4} | {'Thresh':>6} | {'Noise':>5} | {'Pos':>4} | {'Neg':>4} | {'Pos%':>5} | {'Flipped':>7}")
print("=" * 100)

for bot in bot_configs:
    bot_data = ratings_df[ratings_df['bot_id'] == bot.bot_id] # get only this bot's ratings
    pos = bot_data['label'].sum() # count how many YES (1s)
    neg = len(bot_data) - pos # the rest are NO (0s)
    pos_pct = pos / len(bot_data) * 100 # percentage of YES ratings
    flipped = (bot_data['label'] != bot_data['label_clean']).sum() # how many got changed by noise
    thresh = bot_data['threshold'].iloc[0] # the actual threshold value that was computed

    print(f"{bot.bot_id:>4} | {bot.core_movie:<28} | {bot.threshold_percentile:>4.0f} | {thresh:>6.2f} | {bot.noise_pct:>4.0%} | {pos:>4} | {neg:>4} | {pos_pct:>4.1f}% | {flipped:>7}")

print("=" * 100)
print(f"\nOverall: {ratings_df['label'].sum()} positives, {len(ratings_df) - ratings_df['label'].sum()} negatives out of {len(ratings_df)} total")
print(f"Overall positive rate: {ratings_df['label'].mean()*100:.1f}%")

 Bot | Core Movie                   | Pctl | Thresh | Noise |  Pos |  Neg |  Pos% | Flipped
   0 | The Godfather                |   85 |   1.32 |   8% |   96 |  404 | 19.2% |      31
   1 | The Shawshank Redemption     |   45 |   1.21 |  20% |  255 |  245 | 51.0% |     109
   2 | Forrest Gump                 |   75 |   1.08 |  10% |  162 |  338 | 32.4% |      43
   3 | Interstellar                 |   60 |   0.74 |  10% |  215 |  285 | 43.0% |      53
   4 | Psycho                       |   80 |   0.81 |  12% |  131 |  369 | 26.2% |      53
   5 | Saving Private Ryan          |   88 |   1.30 |   7% |   95 |  405 | 19.0% |      39
   6 | Titanic                      |   92 |   1.39 |   5% |   65 |  435 | 13.0% |      26
   7 | Some Like It Hot             |   40 |   0.49 |  18% |  278 |  222 | 55.6% |      80
   8 | Coco                         |   70 |   0.70 |  15% |  184 |  316 | 36.8% |      78
   9 | Spider-Man: Into the Spider-Verse |   95 |   1.21 |   6% |   46 |  454 |  9.2% |  

In [ ]:
# Show 5 example ratings from each bot: 3 YES + 2 NO
for bot_id in range(10):
    bot_data = ratings_df[ratings_df['bot_id'] == bot_id] # filter this bot's data
    core = bot_data['core_movie'].iloc[0] # get the core movie name
    thresh = bot_data['threshold'].iloc[0] # get the threshold
    noise = bot_configs[bot_id].noise_pct # get the noise percentage

    pos = bot_data[bot_data['label'] == 1].head(3) # grab 3 movies the bot said YES to
    neg = bot_data[bot_data['label'] == 0].head(2) # grab 2 movies the bot said NO to
    sample = pd.concat([pos, neg]) # combine them

    print(f"\n{'='*55}")
    print(f"Bot {bot_id} | Loves: '{core}'")
    print(f"Threshold: {thresh} | Noise: {noise:.0%}")
    print(f"{'-'*55}")
    for _, row in sample.iterrows(): # for each sample movie
        verdict = "YES" if row['label'] == 1 else "NO" # convert 1/0 to YES/NO
        print(f"  {row['candidate_movie']:<40} -> {verdict}")
    print()


Bot 0 | Loves: 'The Godfather'
Threshold: 1.32 | Noise: 8%
-------------------------------------------------------
  The Conversation                         -> YES
  Changeling                               -> YES
  Ajeossi                                  -> YES
  The Game                                 -> NO
  Per qualche dollaro in più               -> NO


Bot 1 | Loves: 'The Shawshank Redemption'
Threshold: 1.21 | Noise: 20%
-------------------------------------------------------
  Nelyubov                                 -> YES
  The Count of Monte Cristo                -> YES
  The Machinist                            -> YES
  Charade                                  -> NO
  Knockin' on Heaven's Door                -> NO


Bot 2 | Loves: 'Forrest Gump'
Threshold: 1.08 | Noise: 10%
-------------------------------------------------------
  Lilja 4-ever                             -> YES
  Three Billboards Outside Ebbing, Missouri -> YES
  Rain Man                               

In [ ]:
# ============================================================
# Part 2: Train a Logistic Regression classifier for each bot
# ============================================================
# For each bot we take its 500 ratings, split into train/test,
# and train a Logistic Regression to predict YES/NO from the
# similarity score.

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

bot_models = {} # will store the trained model + test data for each bot

for bot_id in range(10):

    # get this bot's 500 ratings
    bot_data = ratings_df[ratings_df['bot_id'] == bot_id]

    # X = feature (similarity score), y = target (label: 1=YES, 0=NO)
    X = bot_data[['similarity']].values # reshape to 2D array for sklearn
    y = bot_data['label'].values # the noisy label

    # split: 80% train (400 ratings), 20% test (100 ratings)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # train a Logistic Regression model
    model = LogisticRegression(random_state=42)
    model.fit(X_train, y_train)

    # predict on the test set
    y_pred = model.predict(X_test)

    # store everything for later use
    bot_models[bot_id] = {
        'model': model,
        'X_test': X_test,
        'y_test': y_test,
        'y_pred': y_pred,
        'bot_data': bot_data
    }

    # print a quick summary
    acc = accuracy_score(y_test, y_pred)
    core = bot_data['core_movie'].iloc[0]
    print(f"Bot {bot_id} ({core}): trained on {len(X_train)} samples, "
          f"tested on {len(X_test)} samples, accuracy = {acc:.2%}")

Bot 0 (The Godfather): trained on 400 samples, tested on 100 samples, accuracy = 82.00%
Bot 1 (The Shawshank Redemption): trained on 400 samples, tested on 100 samples, accuracy = 74.00%
Bot 2 (Forrest Gump): trained on 400 samples, tested on 100 samples, accuracy = 88.00%
Bot 3 (Interstellar): trained on 400 samples, tested on 100 samples, accuracy = 80.00%
Bot 4 (Psycho): trained on 400 samples, tested on 100 samples, accuracy = 80.00%
Bot 5 (Saving Private Ryan): trained on 400 samples, tested on 100 samples, accuracy = 87.00%
Bot 6 (Titanic): trained on 400 samples, tested on 100 samples, accuracy = 97.00%
Bot 7 (Some Like It Hot): trained on 400 samples, tested on 100 samples, accuracy = 82.00%
Bot 8 (Coco): trained on 400 samples, tested on 100 samples, accuracy = 70.00%
Bot 9 (Spider-Man: Into the Spider-Verse): trained on 400 samples, tested on 100 samples, accuracy = 86.00%


In [ ]:
# Evaluation table: Accuracy, Precision, Recall, F1 for each bot
print("=" * 90)
print(f"{'Bot':>4} | {'Core Movie':<32} | {'Acc':>6} | {'Prec':>6} | {'Recall':>6} | {'F1':>6}")
print("=" * 90)

for bot_id in range(10):
    y_test = bot_models[bot_id]['y_test'] # true labels
    y_pred = bot_models[bot_id]['y_pred'] # predicted labels
    core = bot_models[bot_id]['bot_data']['core_movie'].iloc[0] # core movie name

    acc = accuracy_score(y_test, y_pred) # how many predictions are correct overall
    prec = precision_score(y_test, y_pred, zero_division=0) # of all predicted YES, how many are correct
    rec = recall_score(y_test, y_pred, zero_division=0) # of all actual YES, how many did we catch
    f1 = f1_score(y_test, y_pred, zero_division=0) # harmonic mean of precision and recall

    print(f"{bot_id:>4} | {core:<32} | {acc:>5.1%} | {prec:>5.1%} | {rec:>5.1%} | {f1:>5.1%}")

print("=" * 90)

 Bot | Core Movie                       |    Acc |   Prec | Recall |     F1
   0 | The Godfather                    | 82.0% | 75.0% | 15.0% | 25.0%
   1 | The Shawshank Redemption         | 74.0% | 80.4% | 68.5% | 74.0%
   2 | Forrest Gump                     | 88.0% | 95.0% | 63.3% | 76.0%
   3 | Interstellar                     | 80.0% | 87.9% | 64.4% | 74.4%
   4 | Psycho                           | 80.0% | 76.9% | 37.0% | 50.0%
   5 | Saving Private Ryan              | 87.0% | 85.7% | 33.3% | 48.0%
   6 | Titanic                          | 97.0% | 100.0% | 57.1% | 72.7%
   7 | Some Like It Hot                 | 82.0% | 76.1% | 96.2% | 85.0%
   8 | Coco                             | 70.0% | 88.2% | 34.9% | 50.0%
   9 | Spider-Man: Into the Spider-Verse | 86.0% |  0.0% |  0.0% |  0.0%


In [ ]:
# Detailed classification report for each bot
for bot_id in range(10):
    y_test = bot_models[bot_id]['y_test']
    y_pred = bot_models[bot_id]['y_pred']
    core = bot_models[bot_id]['bot_data']['core_movie'].iloc[0]

    print(f"\n{'='*55}")
    print(f"Bot {bot_id} | Core: '{core}'")
    print(f"{'='*55}")
    # classification_report shows precision, recall, f1 for EACH class (NO and YES)
    print(classification_report(y_test, y_pred, target_names=['NO', 'YES'], zero_division=0))


Bot 0 | Core: 'The Godfather'
              precision    recall  f1-score   support

          NO       0.82      0.99      0.90        80
         YES       0.75      0.15      0.25        20

    accuracy                           0.82       100
   macro avg       0.79      0.57      0.57       100
weighted avg       0.81      0.82      0.77       100


Bot 1 | Core: 'The Shawshank Redemption'
              precision    recall  f1-score   support

          NO       0.69      0.80      0.74        46
         YES       0.80      0.69      0.74        54

    accuracy                           0.74       100
   macro avg       0.74      0.74      0.74       100
weighted avg       0.75      0.74      0.74       100


Bot 2 | Core: 'Forrest Gump'
              precision    recall  f1-score   support

          NO       0.86      0.99      0.92        70
         YES       0.95      0.63      0.76        30

    accuracy                           0.88       100
   macro avg       0.91  

In [ ]:
# Show 5 example test predictions for each bot
for bot_id in range(10):
    y_test = bot_models[bot_id]['y_test']
    y_pred = bot_models[bot_id]['y_pred']
    bot_data = bot_models[bot_id]['bot_data']
    core = bot_data['core_movie'].iloc[0]

    # get the test set indices so we can look up movie names
    test_indices = bot_data.iloc[
        train_test_split(range(len(bot_data)), test_size=0.2, random_state=42)[1]
    ].reset_index(drop=True)

    print(f"\n{'='*65}")
    print(f"Bot {bot_id} | Core: '{core}'")
    print(f"{'-'*65}")
    print(f"  {'Movie':<35} {'True':>6} {'Pred':>6} {'Result':>8}")
    print(f"  {'-'*57}")

    for i in range(min(5, len(y_test))): # show first 5 test samples
        movie_name = test_indices.iloc[i]['candidate_movie']
        true = "YES" if y_test[i] == 1 else "NO" # actual label
        pred = "YES" if y_pred[i] == 1 else "NO" # model's prediction
        result = "OK" if y_test[i] == y_pred[i] else "WRONG" # correct or not

        print(f"  {movie_name:<35} {true:>6} {pred:>6} {result:>8}")
    print()


Bot 0 | Core: 'The Godfather'
-----------------------------------------------------------------
  Movie                                 True   Pred   Result
  ---------------------------------------------------------
  The Taking of Pelham One Two Three      NO     NO       OK
  Gifted                                 YES     NO    WRONG
  Annie Hall                              NO     NO       OK
  Tenkû no shiro Rapyuta                  NO     NO       OK
  Apocalypse Now                         YES    YES       OK


Bot 1 | Core: 'The Shawshank Redemption'
-----------------------------------------------------------------
  Movie                                 True   Pred   Result
  ---------------------------------------------------------
  The Killing                            YES    YES       OK
  Inside Man                             YES    YES       OK
  American Beauty                        YES    YES       OK
  The Naked Gun: From the Files of Police Squad!     NO     NO  